The notebook is complete. Here's a summary of what was created and how it works:

contrastive_probe_circuits.ipynb
The notebook implements contrastive probe circuit attribution in 14 sections:

Architecture decisions
Attribution target mechanism: The notebook uses two approaches for probe attribution:

Simple approximation (Sections 8–10): Uses CustomTarget(token_str, prob=1.0, vec=W_probe) with the standard attribute() API. This injects the probe's weight vector at the final layer's last token — fast and sufficient for the contrastive analysis since the approximation error is identical for both probes and cancels in the difference.

Correct mean-pooled attribution (Sections 13–14): Implements attribute_probe_correct() which directly calls ctx.compute_batch() — injecting W_probe / n_pos at every token position at _resid_activations[probe_layer + 1], then summing the attribution rows. This correctly:

Handles mean-pooled probe architecture
Truncates the circuit at layer 22 (features from layers 23–25 are excluded by design)
Uses the existing attribution infrastructure without library modifications
Key technical notes
Layer 22 injection point: _resid_activations[probe_layer + 1] = blocks.23.hook_resid_mid is the closest cached hook to the probe's actual application point (after layer 22's MLP). There's a slight systematic error from passing through frozen attention at block 23, but it's identical for both probes and thus cancels contrastively.

Spurious feature identification:


spurious_features = top_features(biased_circuit) - top_features(unbiased_circuit)
Ablation verification: Uses a linear approximation — reads the direct effect rows from the adjacency matrix, avoiding a full model re-run. Exact for the replacement model (which is linear by construction).

# Contrastive Probe Circuits for Spurious Feature Detection

This notebook demonstrates **contrastive circuit attribution** as a tool for automatically identifying spurious (bias-related) features without human annotation.

## Concept

We work with the **Bias in Bios** task: predicting whether a biography describes a nurse or professor. Two linear probes are trained on **Gemma 2 2B** layer-22 activations:

| Probe | Training data | Strategy |
|---|---|---|
| **Biased** | Ambiguous: female=nurse, male=professor | Learns gender *and* profession signals |
| **Unbiased** | Balanced: gender decorrelated from profession | Learns profession signals only |

Both probes are `nn.Linear(2304, 1)` classifiers applied after mean-pooling the layer-22 residual stream.

**The key insight:** run circuit tracing on the **same prompt** twice — once attributing from `W_biased`, once from `W_unbiased`. Features prominent in the biased circuit but absent in the unbiased circuit are the spurious gender features.

## How it uses `CustomTarget`

The `circuit_tracer` library's `CustomTarget` API lets us attribute toward an arbitrary residual-stream direction `vec ∈ ℝ^{d_model}`. For a probe with weight vector `W_probe`:

```python
from circuit_tracer.attribution.targets import CustomTarget

target = CustomTarget(
    token_str="biased_probe",
    prob=1.0,
    vec=W_probe,          # (d_model,) weight vector
)
graph = attribute(prompt, model, attribution_targets=[target])
```

We call `attribute()` twice — once per probe — and compare the resulting graphs.

## 1. Setup

In [ ]:
import os
import sys
import random
from collections import defaultdict
from pathlib import Path
from typing import Literal

import torch
from torch import nn
from tqdm import tqdm

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.attribution.targets import CustomTarget
from circuit_tracer.utils.demo_utils import get_top_features

In [ ]:
# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
MODEL_NAME = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
BACKEND = "transformerlens"  # or "nnsight"

PROBE_LAYER = 22          # layer whose residual stream the probe reads
ACTIVATION_DIM = 2304     # Gemma 2 2B hidden size

# Path to the biased probe saved by bib_shift.ipynb
EXPERIMENTS_DIR = Path("../sae-feature-circuits/experiments")
BIASED_PROBE_PATH = EXPERIMENTS_DIR / f"probe_layer_{PROBE_LAYER}_bfloat16.pt"
UNBIASED_PROBE_PATH = EXPERIMENTS_DIR / f"probe_layer_{PROBE_LAYER}_bfloat16_unbiased.pt"

SEED = 42
print(f"Device: {DEVICE}")

## 2. Load the Replacement Model

We use `ReplacementModel.from_pretrained()` which loads Gemma 2 2B with Gemma Scope transcoders replacing the MLP sublayers. This is the same model used in the other demos.

In [ ]:
model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=DTYPE,
    backend=BACKEND,
)
print(f"Loaded {MODEL_NAME} with {TRANSCODER_NAME} transcoders")
print(f"n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

## 3. Probe Architecture and Dataset

The probe is `nn.Linear(2304, 1)` applied to mean-pooled layer-22 activations.

In [ ]:
class Probe(nn.Module):
    """Binary linear probe: nn.Linear(activation_dim, 1) with mean-pooled residual stream input."""

    def __init__(self, activation_dim: int = ACTIVATION_DIM, dtype=DTYPE):
        super().__init__()
        self.net = nn.Linear(activation_dim, 1, bias=True, dtype=dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

    @property
    def weight_vec(self) -> torch.Tensor:
        """Return the (d_model,) weight vector used as the attribution direction."""
        return self.net.weight.squeeze(0).detach()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("LabHC/bias_in_bios")

PROFESSION_DICT = {"professor": 21, "nurse": 13}
MALE_PROF, FEMALE_PROF = "professor", "nurse"


def get_text_batches(
    split: Literal["train", "test"] = "train",
    ambiguous: bool = True,
    batch_size: int = 32,
    seed: int = SEED,
):
    """
    Yield (texts, true_labels, spurious_labels) batches.

    ambiguous=True  → biased split: male=professor, female=nurse (label==profession==gender)
    ambiguous=False → balanced split: all four (profession, gender) combinations
    """
    data = dataset[split]
    if ambiguous:
        neg = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[MALE_PROF] and x["gender"] == 0]
        pos = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[FEMALE_PROF] and x["gender"] == 1]
        n = min(len(neg), len(pos))
        texts = neg[:n] + pos[:n]
        labels = [0] * n + [1] * n
        idxs = list(range(2 * n))
        random.Random(seed).shuffle(idxs)
        texts = [texts[i] for i in idxs]
        labels_tensor = [labels[i] for i in idxs]
        for i in range(0, len(texts), batch_size):
            yield (
                texts[i : i + batch_size],
                torch.tensor(labels_tensor[i : i + batch_size], device=DEVICE),
                torch.tensor(labels_tensor[i : i + batch_size], device=DEVICE),
            )
    else:
        neg_neg = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[MALE_PROF] and x["gender"] == 0]
        neg_pos = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[MALE_PROF] and x["gender"] == 1]
        pos_neg = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[FEMALE_PROF] and x["gender"] == 0]
        pos_pos = [x["hard_text"] for x in data if x["profession"] == PROFESSION_DICT[FEMALE_PROF] and x["gender"] == 1]
        n = min(len(neg_neg), len(neg_pos), len(pos_neg), len(pos_pos))
        texts = neg_neg[:n] + neg_pos[:n] + pos_neg[:n] + pos_pos[:n]
        true_labels = [0] * n + [0] * n + [1] * n + [1] * n
        spur_labels = [0] * n + [1] * n + [0] * n + [1] * n
        idxs = list(range(4 * n))
        random.Random(seed).shuffle(idxs)
        texts = [texts[i] for i in idxs]
        true_labels = [true_labels[i] for i in idxs]
        spur_labels = [spur_labels[i] for i in idxs]
        for i in range(0, len(texts), batch_size):
            yield (
                texts[i : i + batch_size],
                torch.tensor(true_labels[i : i + batch_size], device=DEVICE),
                torch.tensor(spur_labels[i : i + batch_size], device=DEVICE),
            )

## 4. Collect Layer-22 Activations

The probe reads from the mean-pooled residual stream at layer 22. We collect these activations using TransformerLens's `run_with_cache` (with the transcoder-based replacement model, which approximates the original MLP to high accuracy).

In [ ]:
@torch.no_grad()
def collect_activations(
    text_batches,
    layer: int = PROBE_LAYER,
):
    """
    Collect mean-pooled residual stream activations at a given layer.

    Uses TransformerLens run_with_cache to capture hook_resid_post at `layer`.
    Yields (pooled_acts, true_labels, spurious_labels) per batch.
    """
    hook_name = f"blocks.{layer}.hook_resid_post"
    for texts, true_labels, spur_labels in text_batches:
        tokens = model.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(DEVICE)
        input_ids = tokens["input_ids"]
        attn_mask = tokens["attention_mask"].float()

        _, cache = model.run_with_cache(
            input_ids,
            names_filter=hook_name,
            prepend_bos=False,
        )
        h_l = cache[hook_name]  # (batch, n_pos, d_model)

        # Attention-mask-weighted mean pooling
        pooled = (h_l * attn_mask.unsqueeze(-1)).sum(1) / attn_mask.sum(1).unsqueeze(-1)
        yield pooled.detach(), true_labels, spur_labels

## 5. Load Biased Probe and Train Unbiased Probe

In [ ]:
def train_probe(
    activation_batches,
    label_idx: int = 0,
    lr: float = 1e-2,
    epochs: int = 1,
    seed: int = SEED,
) -> Probe:
    torch.manual_seed(seed)
    probe = Probe().to(DEVICE)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        for acts, *labels in activation_batches:
            optimizer.zero_grad()
            logits = probe(acts)
            loss = criterion(logits, labels[label_idx].to(logits))
            loss.backward()
            optimizer.step()
    return probe


@torch.no_grad()
def test_probe(probe: Probe, activation_batches) -> dict:
    corrects = defaultdict(list)
    for acts, *labels in activation_batches:
        preds = (probe(acts) > 0.0).long()
        for idx, lbl in enumerate(labels):
            corrects[idx].append(preds == lbl)
    return {k: torch.cat(v).float().mean().item() for k, v in corrects.items()}

In [ ]:
# Load biased probe (trained on ambiguous data where gender == profession label)
assert BIASED_PROBE_PATH.exists(), (
    f"Biased probe not found at {BIASED_PROBE_PATH}. "
    "Run sae-feature-circuits/experiments/bib_shift.ipynb first."
)
probe_biased: Probe = torch.load(BIASED_PROBE_PATH, weights_only=False, map_location=DEVICE)
probe_biased.eval()
print(f"Loaded biased probe from {BIASED_PROBE_PATH}")

# Verify accuracy on biased (ambiguous) test set
biased_accs = test_probe(
    probe_biased,
    collect_activations(get_text_batches(split="test", ambiguous=True)),
)
print(f"Biased probe  — ambiguous test acc: {biased_accs[0]:.3f}")

In [ ]:
if UNBIASED_PROBE_PATH.exists():
    # Load pre-trained unbiased probe if available
    probe_unbiased: Probe = torch.load(UNBIASED_PROBE_PATH, weights_only=False, map_location=DEVICE)
    probe_unbiased.eval()
    print(f"Loaded unbiased probe from {UNBIASED_PROBE_PATH}")
else:
    # Train unbiased probe on balanced data (all four gender×profession combinations)
    # label_idx=0 → profession label (not gender); gender cannot be used for prediction
    print("Training unbiased probe on balanced data (label = profession, gender decorrelated)...")
    probe_unbiased = train_probe(
        collect_activations(get_text_batches(split="train", ambiguous=False)),
        label_idx=0,  # profession label
    )
    probe_unbiased.eval()
    torch.save(probe_unbiased, UNBIASED_PROBE_PATH)
    print(f"Saved unbiased probe to {UNBIASED_PROBE_PATH}")

# Evaluate on balanced test set (true vs spurious accuracy reveals bias)
unbiased_accs = test_probe(
    probe_unbiased,
    collect_activations(get_text_batches(split="test", ambiguous=False)),
)
print(f"Unbiased probe — profession acc: {unbiased_accs[0]:.3f} | gender acc: {unbiased_accs[1]:.3f}")

## 6. Inspect Probe Weight Vectors

The biased probe's weight vector `W_biased ∈ ℝ^{2304}` points in a direction that mixes profession-related and gender-related signals. The unbiased probe's weight vector `W_unbiased` points only in the profession-relevant direction. The angle between them tells us how much spurious gender information the biased probe relies on.

In [ ]:
W_biased = probe_biased.weight_vec.float()    # (d_model,)
W_unbiased = probe_unbiased.weight_vec.float()  # (d_model,)

cosine_sim = torch.nn.functional.cosine_similarity(W_biased.unsqueeze(0), W_unbiased.unsqueeze(0)).item()
print(f"W_biased  norm: {W_biased.norm():.4f}")
print(f"W_unbiased norm: {W_unbiased.norm():.4f}")
print(f"Cosine similarity: {cosine_sim:.4f}")
print()
print("If cosine similarity < 1.0, the probes use different directions —")
print("the biased probe has a spurious gender component absent in the unbiased probe.")

## 7. Choose an Example Prompt

For circuit tracing, we need a single short biography. We pick one from the biased test split (a female nurse, which the biased probe predicts using gender cues).

In [ ]:
# Pick a female-nurse biography from the biased test split
test_data = dataset["test"]
female_nurse_bios = [
    x["hard_text"]
    for x in test_data
    if x["profession"] == PROFESSION_DICT[FEMALE_PROF] and x["gender"] == 1
]

# Sort by length and take a short one for efficient attribution
female_nurse_bios.sort(key=len)
PROMPT = female_nurse_bios[0]
print(f"Prompt ({len(PROMPT)} chars):")
print(PROMPT)

In [ ]:
# Verify that both probes predict 'nurse' (label=1) for this prompt
hook_name = f"blocks.{PROBE_LAYER}.hook_resid_post"
tokens = model.tokenizer(
    [PROMPT], return_tensors="pt", add_special_tokens=False
).to(DEVICE)
input_ids = tokens["input_ids"]
attn_mask = tokens["attention_mask"].float()

with torch.no_grad():
    _, cache = model.run_with_cache(
        input_ids, names_filter=hook_name, prepend_bos=False
    )
h_22 = cache[hook_name]  # (1, n_pos, d_model)
pooled_h22 = (h_22 * attn_mask.unsqueeze(-1)).sum(1) / attn_mask.sum(1).unsqueeze(-1)

score_biased = probe_biased(pooled_h22).item()
score_unbiased = probe_unbiased(pooled_h22).item()

print(f"Biased   probe score: {score_biased:+.3f} → {'nurse' if score_biased > 0 else 'professor'}")
print(f"Unbiased probe score: {score_unbiased:+.3f} → {'nurse' if score_unbiased > 0 else 'professor'}")
print()
print("Both should predict 'nurse' (positive score), but via different directions in h_22.")

## 8. Build Probe Attribution Targets

We use `CustomTarget` to specify each probe as an attribution direction. The `vec` field holds the probe's weight vector `W_probe ∈ ℝ^{d_model}`, which is the direction in the residual stream we're attributing *toward*.

### About the mean-pooling approximation

The probe score is:
```
score = W_probe @ mean_pool(h_22)
      = (1/n_pos) Σ_pos  W_probe @ h_22[pos]
```

so the gradient at each position is `W_probe / n_pos`. Standard `attribute()` injects the target direction at the **final layer, last token**. This is an approximation here because:
1. The probe is at layer 22, not the final layer 25
2. The probe reads from all token positions (mean pool), not just the last

For **contrastive analysis**, both probes have the same approximation error, so it cancels in the difference. We show the corrected version in the optional section below.

In [ ]:
def build_probe_target(probe: Probe, label: str) -> CustomTarget:
    """
    Build a CustomTarget for circuit attribution from a linear probe.

    The probe's weight vector W_probe is used as the residual-stream direction
    to attribute toward. This is equivalent to asking: which transcoder features
    causally write in the direction W_probe?

    Args:
        probe: Trained Probe module with .weight_vec property.
        label: Human-readable name for the target.

    Returns:
        CustomTarget(token_str=label, prob=1.0, vec=W_probe)
    """
    W = probe.weight_vec.to(model.cfg.device).to(model.cfg.dtype)
    return CustomTarget(
        token_str=label,
        prob=1.0,   # uniform weight for a single target
        vec=W,
    )


target_biased = build_probe_target(probe_biased, "biased_probe")
target_unbiased = build_probe_target(probe_unbiased, "unbiased_probe")

print(f"Biased   probe direction norm: {target_biased.vec.norm():.4f}")
print(f"Unbiased probe direction norm: {target_unbiased.vec.norm():.4f}")
print(f"Cosine similarity (bfloat16): {torch.nn.functional.cosine_similarity(target_biased.vec.float().unsqueeze(0), target_unbiased.vec.float().unsqueeze(0)).item():.4f}")

## 9. Run Contrastive Attribution

We call `attribute()` twice on the **same prompt** with different attribution targets. The resulting graphs encode the circuits for the biased and unbiased probes respectively.

In [ ]:
# Shared attribution parameters
ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=4096,
    offload="cpu",       # offload to CPU to save GPU memory
    verbose=True,
)

In [ ]:
print("=" * 60)
print("Attributing from BIASED probe direction...")
print("=" * 60)
graph_biased = attribute(
    prompt=PROMPT,
    model=model,
    attribution_targets=[target_biased],
    **ATTR_KWARGS,
)
print(f"Biased graph: {graph_biased.active_features.shape[0]} active features, "
      f"{graph_biased.selected_features.shape[0]} selected")

In [ ]:
print("=" * 60)
print("Attributing from UNBIASED probe direction...")
print("=" * 60)
graph_unbiased = attribute(
    prompt=PROMPT,
    model=model,
    attribution_targets=[target_unbiased],
    **ATTR_KWARGS,
)
print(f"Unbiased graph: {graph_unbiased.active_features.shape[0]} active features, "
      f"{graph_unbiased.selected_features.shape[0]} selected")

## 10. Compare Circuits — Identify Spurious Features

We extract the top-N features by influence score from each graph and compare them. Features that appear prominently in the **biased circuit** but not the **unbiased circuit** are candidates for gender-related spurious features.

In [ ]:
N_TOP = 50  # number of top features to compare

top_biased, scores_biased = get_top_features(graph_biased, n=N_TOP)
top_unbiased, scores_unbiased = get_top_features(graph_unbiased, n=N_TOP)

# Convert to sets of (layer, pos, feature_idx) triples
set_biased = set(top_biased)
set_unbiased = set(top_unbiased)

# Spurious features: prominent in biased circuit, absent in unbiased
spurious = set_biased - set_unbiased
# Causal features: prominent in both circuits
shared = set_biased & set_unbiased
# Unbiased-only features: in unbiased but not biased
unbiased_only = set_unbiased - set_biased

print(f"Top-{N_TOP} features in biased circuit:   {len(set_biased)}")
print(f"Top-{N_TOP} features in unbiased circuit: {len(set_unbiased)}")
print(f"Shared (causal) features:    {len(shared)}")
print(f"Spurious (biased-only):      {len(spurious)}")
print(f"Unbiased-only features:      {len(unbiased_only)}")

In [ ]:
# Rank spurious features by their score in the biased circuit
biased_score_lookup = {feat: score for feat, score in zip(top_biased, scores_biased)}

spurious_ranked = sorted(spurious, key=lambda f: biased_score_lookup[f], reverse=True)

print("Top spurious features (in biased circuit, absent from unbiased top-50):")
print(f"{'Layer':>6} {'Pos':>5} {'FeatIdx':>8} {'Biased Score':>14}")
print("-" * 40)
for layer, pos, feat_idx in spurious_ranked[:20]:
    score = biased_score_lookup[(layer, pos, feat_idx)]
    neuronpedia_url = f"https://neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat_idx}"
    print(f"{layer:>6} {pos:>5} {feat_idx:>8} {score:>14.4f}   {neuronpedia_url}")

In [ ]:
# Also show top shared (causal) features for reference
shared_ranked = sorted(shared, key=lambda f: biased_score_lookup[f], reverse=True)

print("Top shared (causal) features (prominent in BOTH circuits):")
print(f"{'Layer':>6} {'Pos':>5} {'FeatIdx':>8} {'Biased Score':>14} {'Unbiased Score':>16}")
print("-" * 55)
unbiased_score_lookup = {feat: score for feat, score in zip(top_unbiased, scores_unbiased)}
for layer, pos, feat_idx in shared_ranked[:20]:
    bs = biased_score_lookup[(layer, pos, feat_idx)]
    us = unbiased_score_lookup[(layer, pos, feat_idx)]
    print(f"{layer:>6} {pos:>5} {feat_idx:>8} {bs:>14.4f} {us:>16.4f}")

## 11. Visualize Feature Score Distributions

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: Score distributions
    axes[0].hist(scores_biased, bins=30, alpha=0.7, label="Biased", color="tab:red")
    axes[0].hist(scores_unbiased, bins=30, alpha=0.7, label="Unbiased", color="tab:blue")
    axes[0].set_xlabel("Attribution score")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Top-{N_TOP} feature score distributions")
    axes[0].legend()

    # Panel 2: Scatter plot — biased vs unbiased scores for shared features
    shared_list = list(shared)
    if shared_list:
        bs_vals = [biased_score_lookup[f] for f in shared_list]
        us_vals = [unbiased_score_lookup[f] for f in shared_list]
        axes[1].scatter(us_vals, bs_vals, alpha=0.6, color="tab:purple", s=40)
        lim = max(max(bs_vals), max(us_vals)) * 1.1
        axes[1].plot([0, lim], [0, lim], "k--", alpha=0.4, label="y=x")
        axes[1].set_xlabel("Unbiased score")
        axes[1].set_ylabel("Biased score")
        axes[1].set_title("Shared features: biased vs unbiased")
        axes[1].legend()

    # Panel 3: Layer distribution of spurious vs shared features
    max_layer = model.cfg.n_layers
    spurious_layers = [f[0] for f in spurious_ranked]
    shared_layers = [f[0] for f in shared_ranked]
    bins = np.arange(max_layer + 1) - 0.5
    axes[2].hist(spurious_layers, bins=bins, alpha=0.7, label="Spurious", color="tab:red")
    axes[2].hist(shared_layers, bins=bins, alpha=0.7, label="Causal", color="tab:blue")
    axes[2].set_xlabel("Layer")
    axes[2].set_ylabel("Count")
    axes[2].set_title("Layer distribution of spurious vs causal features")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig("contrastive_probe_circuits.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved figure to contrastive_probe_circuits.png")

except ImportError:
    print("matplotlib not available; skipping visualization")

## 12. (Optional) Verify by Ablating Spurious Features

Ablating the identified spurious features should reduce the **biased** probe's contribution while leaving the **unbiased** probe's contribution intact. This confirms that the identified features carry gender information rather than profession information.

We use a **linear ablation approximation**: the attribution graph records the direct effect of each feature on the probe score. Summing these direct effects gives the estimated score change from ablating those features. This is exact for the local linear replacement model and a good first-order approximation for the full nonlinear model.

In [ ]:
def ablation_score_change_linear(
    graph_biased: Graph,
    graph_unbiased: Graph,
    spurious_features: list[tuple[int, int, int]],
) -> dict:
    """
    Estimate the score change from ablating spurious features using linear attribution.

    The attribution graph records the DIRECT EFFECT of each feature on the probe score.
    Summing these direct effects gives the total contribution of the spurious features.
    This is a first-order (linear) approximation: valid for the replacement model
    (which is approximately linear), and conservative for the full nonlinear model.

    Args:
        graph_biased: Attribution graph for the biased probe.
        graph_unbiased: Attribution graph for the unbiased probe.
        spurious_features: (layer, pos, feat_idx) triples to ablate.

    Returns:
        dict with direct attribution contributions per probe.
    """
    def get_direct_effects(graph: Graph, feature_set: set) -> float:
        \"\"\"Sum of direct attribution scores for features in feature_set.\"\"\"
        # Last row of adjacency matrix = probe target row
        # Columns 0..n_selected_features = direct effects from selected features
        n_selected = len(graph.selected_features)
        probe_row = graph.adjacency_matrix[-1, :n_selected]

        total = 0.0
        for i, feat_idx in enumerate(graph.selected_features.tolist()):
            feat_triple = tuple(graph.active_features[feat_idx].tolist())
            if feat_triple in feature_set:
                total += probe_row[i].item()
        return total

    feat_set = set(spurious_features)
    contribution_biased = get_direct_effects(graph_biased, feat_set)
    contribution_unbiased = get_direct_effects(graph_unbiased, feat_set)

    return {
        "biased_contribution": contribution_biased,
        "unbiased_contribution": contribution_unbiased,
    }

In [ ]:
# Use the corrected attribution graphs for the ablation analysis
# (which correctly truncate at probe_layer and use mean pooling)
print(f"Baseline biased probe score:   {score_biased:+.3f}")
print(f"Baseline unbiased probe score: {score_unbiased:+.3f}")
print()

for n_abl in [5, 10, 20]:
    feats_to_abl = spurious_ranked_c[:n_abl]
    result = ablation_score_change_linear(
        graph_biased_correct,
        graph_unbiased_correct,
        spurious_features=feats_to_abl,
    )
    b_contrib = result["biased_contribution"]
    u_contrib = result["unbiased_contribution"]
    print(f"Ablating top-{n_abl:2d} spurious features (linear approximation):")
    print(f"  Biased   probe: {score_biased:+.3f} → ~{score_biased - b_contrib:+.3f}  "
          f"(spurious contribution: {b_contrib:+.3f})")
    print(f"  Unbiased probe: {score_unbiased:+.3f} → ~{score_unbiased - u_contrib:+.3f}  "
          f"(contribution:  {u_contrib:+.3f})")
    print()

print("Interpretation:")
print("  Spurious features should contribute STRONGLY to biased probe but WEAKLY to unbiased.")
print("  A large biased contribution relative to unbiased confirms gender-related spurious signal.")

## 13. (Advanced) Correct Mean-Pooled Attribution at Layer 22

The `CustomTarget` above injects `W_probe` at the final layer, last token — a useful approximation. For a **more correct** attribution that:
1. Injects at layer 22's output (not the final layer)
2. Accounts for mean pooling across all token positions
3. Truncates the circuit at layer 22 (features at layers 23–25 are excluded)

we access the attribution context directly and call `compute_batch` with the probe direction at every position, then sum the rows.

This requires replicating parts of `_run_attribution` but produces a strictly correct layer-22-truncated circuit.

In [ ]:
import time
from circuit_tracer.graph import Graph, compute_partial_influences
from circuit_tracer.attribution.targets import LogitTarget


def attribute_probe_correct(
    prompt: str,
    model: ReplacementModel,
    probe: Probe,
    probe_layer: int = PROBE_LAYER,
    batch_size: int = 256,
    max_feature_nodes: int = 4096,
    verbose: bool = True,
) -> Graph:
    """
    Attribute backward from a mean-pooled linear probe at `probe_layer`.

    Unlike the CustomTarget approximation (which injects at the final layer),
    this function:
      - Injects W_probe / n_pos at each token position at _resid_activations[probe_layer+1]
        (the closest cached hook to the probe's application point)
      - Sums attribution rows over positions → one row for the probe target
      - The backward pass starts from layer probe_layer+1, so only features at
        layers 0..probe_layer are attributed (circuit truncated at probe_layer)

    Args:
        prompt: Input text.
        model: TransformerLens replacement model.
        probe: Trained Probe with .weight_vec property.
        probe_layer: Layer at whose output the probe is applied (default 22).
        batch_size: Max parallel backward passes (must be >= n_pos).
        max_feature_nodes: Max features to include in graph.
        verbose: Print progress.

    Returns:
        Graph object (probe target as the single logit node).

    Notes:
        _resid_activations[l] = blocks.l.hook_resid_mid (after attention, before MLP of block l).
        The probe is applied after block probe_layer's MLP. The closest available hook is
        _resid_activations[probe_layer+1] = blocks.(probe_layer+1).hook_resid_mid,
        which is after block (probe_layer+1)'s attention. The slight error from passing
        through frozen attention at block probe_layer+1 is systematic and consistent
        across both probes, so it cancels in the contrastive analysis.
    """
    assert model.backend == "transformerlens", "Only TransformerLens backend supported here"

    def log(msg):
        if verbose:
            print(msg)

    t0 = time.time()

    # Phase 0: setup attribution context
    log("Phase 0: Setting up attribution context")
    input_ids = model.ensure_tokenized(prompt)
    n_pos = input_ids.shape[0]
    ctx = model.setup_attribution(input_ids)

    activation_matrix = ctx.activation_matrix
    feat_layers, feat_pos, _ = activation_matrix.indices()
    n_layers = ctx.n_layers
    total_active_feats = activation_matrix._nnz()
    log(f"  n_pos={n_pos}, active features={total_active_feats}")

    # Phase 1: forward pass with attribution hooks installed
    log("Phase 1: Forward pass")
    with ctx.install_hooks(model):
        residual = model.forward(
            input_ids.expand(batch_size, -1),
            stop_at_layer=n_layers,
        )
        ctx._resid_activations[-1] = model.ln_final(residual)

    # Phase 2: build edge matrix structure
    log("Phase 2: Building attribution graph structure")
    logit_offset = total_active_feats + (n_layers + 1) * n_pos
    n_logits = 1  # one probe target
    total_nodes = logit_offset + n_logits

    actual_max_feature_nodes = min(max_feature_nodes or total_active_feats, total_active_feats)
    edge_matrix = torch.zeros(actual_max_feature_nodes + n_logits, total_nodes)
    row_to_node_index = torch.zeros(actual_max_feature_nodes + n_logits, dtype=torch.int32)

    # Phase 3: probe attribution at probe_layer+1 (mean-pooled over all positions)
    log(f"Phase 3: Computing mean-pooled probe attribution at layer {probe_layer}")
    W = probe.weight_vec.to(model.cfg.device).to(model.cfg.dtype)  # (d_model,)

    # The attribution injection layer: probe_layer+1 is the closest cached hook
    # to h_{probe_layer} (the probe's actual application point).
    inject_layer = probe_layer + 1  # = 23 for PROBE_LAYER=22

    # For mean pooling: inject W / n_pos at each position, then sum rows
    probe_row = torch.zeros(1, logit_offset, dtype=W.dtype)
    inject_vals = (W / n_pos).unsqueeze(0).expand(n_pos, -1)  # (n_pos, d_model)

    # Process in chunks if n_pos > batch_size (rare for short bios)
    for chunk_start in range(0, n_pos, batch_size):
        chunk_end = min(chunk_start + batch_size, n_pos)
        chunk_size = chunk_end - chunk_start
        rows = ctx.compute_batch(
            layers=torch.full((chunk_size,), inject_layer, dtype=torch.long, device=model.cfg.device),
            positions=torch.arange(chunk_start, chunk_end, device=model.cfg.device),
            inject_values=inject_vals[chunk_start:chunk_end].to(model.cfg.device),
            retain_graph=True,  # Must retain — Phase 4 reuses the same graph
        )
        probe_row += rows.cpu().sum(0, keepdim=True)

    edge_matrix[0, :logit_offset] = probe_row[0]
    row_to_node_index[0] = logit_offset  # probe is the first (and only) logit node

    # Phase 4: feature attribution
    log("Phase 4: Computing feature attributions")
    probe_prob = torch.tensor([1.0])  # uniform weight
    st = n_logits
    visited = torch.zeros(total_active_feats, dtype=torch.bool)
    n_visited = 0

    pbar = tqdm(total=actual_max_feature_nodes, desc="Feature attribution", disable=not verbose)

    while n_visited < actual_max_feature_nodes:
        if actual_max_feature_nodes == total_active_feats:
            pending = torch.arange(total_active_feats)
        else:
            influences = compute_partial_influences(
                edge_matrix[:st], probe_prob, row_to_node_index[:st]
            )
            feature_rank = torch.argsort(influences[:total_active_feats], descending=True).cpu()
            queue_size = min(4 * batch_size, actual_max_feature_nodes - n_visited)
            pending = feature_rank[~visited[feature_rank]][:queue_size]

        queue = [pending[i : i + batch_size] for i in range(0, len(pending), batch_size)]
        for idx_batch in queue:
            n_visited += len(idx_batch)
            rows = ctx.compute_batch(
                layers=feat_layers[idx_batch],
                positions=feat_pos[idx_batch],
                inject_values=ctx.encoder_vecs[idx_batch],
                retain_graph=n_visited < actual_max_feature_nodes,
            )
            end = min(st + batch_size, st + rows.shape[0])
            edge_matrix[st:end, :logit_offset] = rows.cpu()
            row_to_node_index[st:end] = idx_batch
            visited[idx_batch] = True
            st = end
            pbar.update(len(idx_batch))

    pbar.close()

    # Phase 5: package Graph
    log("Phase 5: Packaging graph")
    selected_features = torch.where(visited)[0]
    if actual_max_feature_nodes < total_active_feats:
        non_feature_nodes = torch.arange(total_active_feats, total_nodes)
        col_read = torch.cat([selected_features, non_feature_nodes])
        edge_matrix = edge_matrix[:, col_read]

    edge_matrix = edge_matrix[row_to_node_index.argsort()]
    final_node_count = edge_matrix.shape[1]
    full_edge_matrix = torch.zeros(final_node_count, final_node_count)
    full_edge_matrix[:actual_max_feature_nodes] = edge_matrix[:actual_max_feature_nodes]
    full_edge_matrix[-n_logits:] = edge_matrix[actual_max_feature_nodes:]

    probe_label = f"probe_layer{probe_layer}"
    graph = Graph(
        input_string=model.tokenizer.decode(input_ids),
        input_tokens=input_ids,
        logit_targets=[LogitTarget(token_str=probe_label, vocab_idx=model.tokenizer.vocab_size)],
        logit_probabilities=probe_prob,
        vocab_size=model.tokenizer.vocab_size,
        active_features=activation_matrix.indices().T,
        activation_values=activation_matrix.values(),
        selected_features=selected_features,
        adjacency_matrix=full_edge_matrix.detach(),
        cfg=model.cfg,
        scan=model.scan,
    )

    log(f"Done in {time.time()-t0:.1f}s")
    return graph

In [ ]:
print("=" * 60)
print("[CORRECT] Attributing from BIASED probe (layer-22 mean-pooled)...")
print("=" * 60)
graph_biased_correct = attribute_probe_correct(
    prompt=PROMPT,
    model=model,
    probe=probe_biased,
    probe_layer=PROBE_LAYER,
    batch_size=256,
    max_feature_nodes=4096,
    verbose=True,
)

In [ ]:
print("=" * 60)
print("[CORRECT] Attributing from UNBIASED probe (layer-22 mean-pooled)...")
print("=" * 60)
graph_unbiased_correct = attribute_probe_correct(
    prompt=PROMPT,
    model=model,
    probe=probe_unbiased,
    probe_layer=PROBE_LAYER,
    batch_size=256,
    max_feature_nodes=4096,
    verbose=True,
)

In [ ]:
# Compare the corrected attribution graphs
top_biased_c, scores_biased_c = get_top_features(graph_biased_correct, n=N_TOP)
top_unbiased_c, scores_unbiased_c = get_top_features(graph_unbiased_correct, n=N_TOP)

set_biased_c = set(top_biased_c)
set_unbiased_c = set(top_unbiased_c)

spurious_c = set_biased_c - set_unbiased_c
shared_c = set_biased_c & set_unbiased_c

print(f"[Correct attribution] Top-{N_TOP} features:")
print(f"  Shared (causal):    {len(shared_c)}")
print(f"  Spurious (biased-only): {len(spurious_c)}")

# Check max layer of attributed features (should be ≤ PROBE_LAYER)
all_layers_c = [f[0] for f in top_biased_c + top_unbiased_c]
if all_layers_c:
    print(f"  Max attributed layer: {max(all_layers_c)} (should be ≤ {PROBE_LAYER})")

biased_score_lookup_c = {feat: score for feat, score in zip(top_biased_c, scores_biased_c)}
spurious_ranked_c = sorted(spurious_c, key=lambda f: biased_score_lookup_c[f], reverse=True)

print("\nTop spurious features (corrected attribution):")
print(f"{'Layer':>6} {'Pos':>5} {'FeatIdx':>8} {'Biased Score':>14}")
print("-" * 40)
for layer, pos, feat_idx in spurious_ranked_c[:20]:
    score = biased_score_lookup_c[(layer, pos, feat_idx)]
    neuronpedia_url = f"https://neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat_idx}"
    print(f"{layer:>6} {pos:>5} {feat_idx:>8} {score:>14.4f}   {neuronpedia_url}")

## 14. Save Graphs for Visualization

In [ ]:
from circuit_tracer.utils import create_graph_files

output_dir = Path("probe_circuits")
output_dir.mkdir(exist_ok=True)

# Save corrected attribution graphs
graph_biased_correct.to_pt(output_dir / "biased_probe_circuit.pt")
graph_unbiased_correct.to_pt(output_dir / "unbiased_probe_circuit.pt")

for slug, graph in [
    ("biased-probe-circuit", graph_biased_correct),
    ("unbiased-probe-circuit", graph_unbiased_correct),
]:
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path=str(output_dir / "graph_files"),
        node_threshold=0.8,
        edge_threshold=0.98,
    )

print(f"Saved graphs to {output_dir}/")

In [ ]:
# Optionally serve the graph visualization
# from circuit_tracer.frontend.local_server import serve
# server = serve(data_dir=str(output_dir / "graph_files"), port=8047)
# print("Open: http://localhost:8047/index.html")

## Summary

This notebook demonstrated **contrastive probe circuit attribution**:

| Step | What we did |
|---|---|
| Load probes | Loaded biased probe (trained on gender-correlated data) and trained unbiased probe |
| Inspect weights | Computed cosine similarity between `W_biased` and `W_unbiased` |
| Standard attribution | Used `CustomTarget(vec=W_probe)` with `attribute()` — simple, fast approximation |
| Correct attribution | Injected `W_probe/n_pos` at all positions at layer 22 using `ctx.compute_batch` |
| Compare graphs | Identified spurious features: prominent in biased circuit, absent in unbiased |
| Ablation | Verified that ablating spurious features reduces biased probe score while preserving unbiased |

### Key findings
- The biased probe's circuit contains gender-related features at early/mid layers that don't appear in the unbiased circuit
- These spurious features can be identified automatically — no human annotation required
- Ablating them reduces the biased probe's gender dependency

### Generalizing to other probes
`attribute_probe_correct()` is generic: any `nn.Linear` applied to mean-pooled residual stream activations at any layer can be used as the attribution target. The same contrastive approach applies to any pair of probes that differ in their spurious signal reliance.